In [1]:
import pandas as pd
import yaml

In [2]:
# ----------------------------
# Load Gesture Data
# ----------------------------

with open("static_gestures.yaml", "r") as f:
    gestures = yaml.safe_load(f)["gestures"]

print(gestures)


['OPEN_PALM', 'POINT_INDEX']


In [3]:
# ----------------------------
# Load dataset
# ----------------------------

df = pd.DataFrame()

for gesture in gestures:
    df = pd.concat([df, pd.read_csv(f"dataset/datasets/{gesture}.csv")])

df = df.sample(frac=1).reset_index(drop=True) #shuffling
df.head()

,x_0,y_0,x_1,y_1,x_2,y_2,x_3,y_3,x_4,y_4,...,y_16,x_17,y_17,x_18,y_18,x_19,y_19,x_20,y_20,gesture
0,0.765788,0.325376,0.754970,0.355477,0.735276,0.384121,0.713918,0.411741,0.698303,0.438486,...,0.245217,0.708834,0.251192,0.683427,0.227152,0.667264,0.210972,0.652104,0.197434,OPEN_PALM
1,0.723866,0.597541,0.699317,0.558785,0.689077,0.515764,0.694515,0.486176,0.703876,0.468305,...,0.536339,0.764685,0.505370,0.736262,0.502764,0.731924,0.525547,0.737116,0.538457,POINT_INDEX
2,0.525324,0.454778,0.579920,0.453129,0.623400,0.437066,0.660092,0.428217,0.694284,0.412643,...,0.255681,0.559268,0.394963,0.558891,0.355041,0.567032,0.327793,0.578159,0.301944,OPEN_PALM
3,0.774556,0.877367,0.759392,0.855635,0.752781,0.822461,0.756197,0.809099,0.766799,0.810264,...,0.893489,0.866706,0.875061,0.834553,0.905321,0.807723,0.914311,0.797699,0.915126,POINT_INDEX
4,0.807482,0.565357,0.784386,0.531411,0.776583,0.489109,0.786993,0.458109,0.801973,0.442661,...,0.514201,0.843462,0.480443,0.847143,0.465660,0.840624,0.491174,0.835517,0.509485,POINT_INDEX


In [4]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [5]:
# Separate features and labels
X = df.drop(columns=["gesture"]).values   # shape: (N, 42)
y = df["gesture"].values  # shape: (N,)

In [6]:
# ----------------------------
# Encode labels
# ----------------------------
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Save encoder because you will need it at inference time
joblib.dump(encoder, "models/static_gesture_label_encoder.pkl")

['models/static_gesture_label_encoder.pkl']

In [7]:
# ----------------------------
# Train-test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, shuffle=True
)

In [8]:
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(480, 42) (480,) (120, 42) (120,)


In [9]:
# ----------------------------
# Train model
# ----------------------------
model = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,400
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
# ----------------------------
# Evaluate
# ----------------------------
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)

print(f"\nAccuracy: {acc:.4f}\n")
print(classification_report(y_test, preds, target_names=encoder.classes_))


Accuracy: 0.9750

              precision    recall  f1-score   support

   OPEN_PALM       1.00      0.95      0.97        55
 POINT_INDEX       0.96      1.00      0.98        65

    accuracy                           0.97       120
   macro avg       0.98      0.97      0.97       120
weighted avg       0.98      0.97      0.97       120



In [11]:
# ----------------------------
# Save model
# ----------------------------
joblib.dump(model, "models/static_gesture_classifier.pkl")

print("\nModel saved as static_gesture_classifier.pkl")
print("Label encoder saved as static_gesture_label_encoder.pkl")


Model saved as static_gesture_classifier.pkl
Label encoder saved as static_gesture_label_encoder.pkl
